# Fit LG — Twitch community graphs (`twitch`)

Batch fit and model comparison for **all networks** in this collection.

Six MUSAE Twitch community graphs (`data/twitch/graphs_processed/*_graph.edges`), sorted by |V|.

Pipeline (same as `1-fit-single-net.ipynb`, repeated for each network):

1. Pick `d̂` via AIC (`select_d_ensemble`, candidates `[0,1,2,3]`).
2. Estimate `σ̂` via Layer-2 offset logit at `d̂`.
3. Compare **LG** (at `d̂`) vs **ER / WS / BA** via `GraphModelComparator`.
4. Save per-network artefacts under `runs/twitch/{graph_name}/` and aggregate tables at `runs/twitch/`.

## 1. Discover networks

In [ ]:
import os
for v in ("OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "MKL_NUM_THREADS"):
    os.environ.setdefault(v, "1")

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import pickle
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import networkx as nx

try:
    from IPython.display import display
except ImportError:
    display = print

from logit_graph import (
    GraphModelComparator,
    calculate_graph_attributes,
    estimate_sigma_from_graph,
    select_d_ensemble,
)

SEED = 0
np.random.seed(SEED)

PLATFORM = "twitch"
DATA_ROOT = Path("..") / ".." / "data"
RUN_DIR = Path("runs") / PLATFORM
RUN_DIR.mkdir(parents=True, exist_ok=True)

MIN_NODES = 0
D_CANDIDATES = [0, 1, 2, 3]


def _load_edges(path: Path) -> nx.Graph:
    G = nx.read_edgelist(path, nodetype=int)
    G = nx.Graph(G)
    G.remove_edges_from(nx.selfloop_edges(G))
    if G.number_of_nodes() == 0:
        raise ValueError(f"Empty graph loaded from {path}")
    cc = max(nx.connected_components(G), key=len)
    return nx.convert_node_labels_to_integers(G.subgraph(cc).copy())


def _lg_max_iterations(n: int) -> int:
    if n <= 100:
        return 4000
    if n <= 300:
        return 8000
    if n <= 500:
        return 12000
    if n <= 700:
        return 16000
    return 20000


def discover_graph_files() -> list[Path]:
    paths = sorted(DATA_ROOT.glob("twitch/graphs_processed/*_graph.edges"))
    paths = [p for p in paths if p.suffix == ".edges" and p.is_file()]
    kept = []
    skipped = []
    for p in paths:
        try:
            G = _load_edges(p)
        except (ValueError, OSError, nx.NetworkXError) as exc:
            skipped.append((p.name, str(exc)))
            continue
        if G.number_of_nodes() >= MIN_NODES:
            kept.append((p, G.number_of_nodes()))
    kept.sort(key=lambda x: x[1])
    if skipped:
        print(f"Skipped {len(skipped)} unreadable/empty graphs")
        for name, reason in skipped[:5]:
            print(f"  {name}: {reason}")
        if len(skipped) > 5:
            print(f"  ... and {len(skipped) - 5} more")
    return [p for p, _ in kept]

graph_files = discover_graph_files()
print(f"PLATFORM={PLATFORM}  RUN_DIR={RUN_DIR.resolve()}")
print(f"Found {len(graph_files)} networks (MIN_NODES={MIN_NODES})")
for p in graph_files:
    G = _load_edges(p)
    print(f"  {p.name:>30s}  n={G.number_of_nodes():>5d}  |E|={G.number_of_edges():>7d}")

## 2. Fit all networks

In [ ]:
comparators: list[GraphModelComparator] = []
summary_rows: list[pd.DataFrame] = []
fit_meta_rows: list[dict] = []
failures: list[dict] = []

for i, edge_path in enumerate(graph_files, start=1):
    graph_name = edge_path.stem
    print(f"\n[{i}/{len(graph_files)}] {graph_name}", flush=True)
    try:
        G_real = _load_edges(edge_path)
        n, m = G_real.number_of_nodes(), G_real.number_of_edges()
        adj = nx.to_numpy_array(G_real)

        d_hat, aic_stats = select_d_ensemble(
            graphs=[adj],
            d_candidates=D_CANDIDATES,
            feature_mode="incremental",
            extra_penalty_per_d=0.0,
        )
        sigma_hat = estimate_sigma_from_graph(adj, d_hat, feature_mode="incremental")
        print(f"  AIC d̂={d_hat}  σ̂={sigma_hat:+.4f}  (n={n}, |E|={m})", flush=True)

        lg_params = {
            "max_iterations": _lg_max_iterations(n),
            "patience": 500,
            "edge_delta": None,
            "min_gic_threshold": 5,
            "er_p": 0.05,
            "check_interval": 50,
        }

        comparator = GraphModelComparator(
            d_list=[d_hat],
            lg_params=lg_params,
            other_model_n_runs=2,
            dist_type="KL",
            verbose=False,
            other_models=["ER", "WS", "BA"],
            other_model_grid_points=5,
        ).compare(original_graph=G_real, graph_filepath=graph_name)

        comparators.append(comparator)
        summary = comparator.summary_df.copy()
        summary_rows.append(summary)

        scored = summary[summary["model"] != "Original"].dropna(subset=["gic_value"])
        best = scored.sort_values("gic_value").iloc[0]
        fit_meta_rows.append({
            "graph": graph_name,
            "n_nodes": n,
            "n_edges": m,
            "d_hat": int(d_hat),
            "sigma_hat": float(sigma_hat),
            "best_model": str(best["model"]),
            "best_gic": float(best["gic_value"]),
        })

        net_dir = RUN_DIR / graph_name
        net_dir.mkdir(parents=True, exist_ok=True)

        with open(net_dir / "comparator.pkl", "wb") as f:
            pickle.dump(comparator, f)

        aic_rows = [{
            "d": d,
            "AIC": s["aic"],
            "ll": s["ll"],
            "sigma_hat(d)": s["sigma_hat"],
            "n_obs": int(s["n_obs"]),
            "selected": d == d_hat,
        } for d, s in aic_stats.items()]
        pd.DataFrame(aic_rows).set_index("d").round(6).to_csv(net_dir / "aic_table.csv")
        summary.round(6).to_csv(net_dir / "summary.csv", index=False)

        with open(RUN_DIR / f"comparators_{graph_name}.pkl", "wb") as f:
            pickle.dump(comparators, f)
    except Exception as exc:
        import traceback
        print(f"  FAILED {graph_name}: {exc}", flush=True)
        traceback.print_exc()
        failures.append({"graph": graph_name, "error": str(exc)})

if failures:
    pd.DataFrame(failures).to_csv(RUN_DIR / "failures.csv", index=False)
    print(f"\n{len(failures)} network(s) failed — see {RUN_DIR / 'failures.csv'}", flush=True)

if not summary_rows:
    raise RuntimeError("No networks were processed — check DATA_ROOT / MIN_NODES.")

summary_all = pd.concat(summary_rows, ignore_index=True)
fit_meta = pd.DataFrame(fit_meta_rows)
summary_all.to_csv(RUN_DIR / "summary_all.csv", index=False)
fit_meta.to_csv(RUN_DIR / "fit_meta_all.csv", index=False)
print(f"\nDone — {len(summary_rows)}/{len(graph_files)} networks OK. Aggregate tables saved to {RUN_DIR.resolve()}", flush=True)

## 3. Aggregate comparison

In [ ]:
models = ["LG", "ER", "WS", "BA"]
gic_pivot = summary_all[summary_all["model"].isin(models)].pivot_table(
    index="graph_filename", columns="model", values="gic_value", aggfunc="first"
)
gic_pivot = gic_pivot.reindex(columns=models)
gic_pivot.to_csv(RUN_DIR / "gic_pivot.csv")

rank_pivot = gic_pivot.rank(axis=1, method="average")
rank_pivot.to_csv(RUN_DIR / "gic_rank_pivot.csv")

mean_rank = rank_pivot.mean(axis=0).sort_values()
print("Mean GIC rank by model (lower = better):")
print(mean_rank.round(3))

print(gic_pivot.round(3).to_string())
print()
print(rank_pivot.round(1).to_string())

## 4. Summary plots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

mean_rank.plot(kind="bar", ax=axes[0], color="#2b6cb0", edgecolor="white")
axes[0].set_ylabel("Mean GIC rank (lower = better)")
axes[0].set_title("twitch — mean model rank across networks")
axes[0].tick_params(axis="x", rotation=0)
axes[0].grid(axis="y", alpha=0.3)

best_counts = fit_meta["best_model"].value_counts().reindex(models, fill_value=0)
axes[1].bar(best_counts.index, best_counts.values, color="#38a169", edgecolor="white")
axes[1].set_ylabel("# networks")
axes[1].set_title("twitch — best model by GIC count")
axes[1].grid(axis="y", alpha=0.3)

fig.tight_layout()
out_plot = RUN_DIR / "aggregate_model_comparison.png"
fig.savefig(out_plot, dpi=150, bbox_inches="tight")
plt.show()
print(f"Saved {out_plot}")